# bravo11234 - Part D Quantitative Evaluation (Colab GPU)

Runs `evaluate_all.py` on a Colab T4 / A100, producing `partD_quantitative.csv`.

## Prerequisites (do this **once** before running the notebook)

1. On your computer: compress the `bravo11234/` folder (right-click > Send to > Compressed (zipped) folder) **or** just drag-and-drop the whole `bravo11234` folder into Google Drive.
2. Put it at Google Drive root, so in Drive the path looks like:

   ```
   My Drive/
   └── bravo11234/
       ├── CODE/
       └── MISC/
           ├── data/             # train.csv / val.csv / test.csv / manual_annotation_200.csv
           └── saved_model/      # roberta_base / bert_clean / bert_aug / distilbert / naive_bayes
   ```

3. Runtime > Change runtime type > Hardware accelerator = **T4 GPU** (free) or higher.
4. Then run the cells below top-to-bottom.

## 1. Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

Mounted at /content/drive


## 2. Locate project + sanity-check structure

If `PROJECT` doesn't match where you dropped the folder, edit it below (e.g. `/content/drive/MyDrive/6713groupwork/bravo11234`).

In [2]:
import os, glob
from pathlib import Path

PROJECT = Path('/content/drive/MyDrive/bravo11234')
assert PROJECT.exists(), f'not found: {PROJECT} - adjust PROJECT to wherever you uploaded the folder.'
os.chdir(PROJECT / 'CODE')
print('cwd =', os.getcwd())

print('\n--- data ---')
for p in sorted((PROJECT / 'MISC' / 'data').glob('*.csv')):
    print(f'  {p.name:35s} {p.stat().st_size/1e6:7.1f} MB')

print('\n--- saved_model ---')
for sub in sorted((PROJECT / 'MISC' / 'saved_model').iterdir()):
    if sub.is_dir():
        sz = sum(p.stat().st_size for p in sub.rglob('*') if p.is_file()) / 1e6
        have_cfg = (sub / 'config.json').exists()
        have_pkl = any(sub.glob('*.pkl'))
        tag = 'HF' if have_cfg else ('NB' if have_pkl else 'EMPTY')
        print(f'  {sub.name:15s} {tag:5s} {sz:7.1f} MB')

cwd = /content/drive/MyDrive/bravo11234/CODE

--- data ---
  manual_annotation_200.csv               0.0 MB
  test.csv                                2.3 MB
  test_deepclean.csv                      2.1 MB
  train.csv                              18.7 MB
  train_deepclean.csv                    17.0 MB
  val.csv                                 2.3 MB

--- saved_model ---
  bert_aug        HF      438.7 MB
  bert_clean      HF      438.7 MB
  distilbert      HF      268.6 MB
  naive_bayes     NB       25.9 MB
  roberta_base    HF      502.0 MB


## 3. Install missing dependencies

Colab already ships with `torch`, `transformers`, `pandas`, `scikit-learn`, `nltk`, so this usually installs nothing new. `joblib` is also there for Naive Bayes.

In [3]:
!pip install -q 'transformers>=4.44,<5' 'accelerate>=0.30' 'datasets>=2.19'
import torch, transformers, sklearn, nltk
print('torch       :', torch.__version__, '| cuda:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('transformers:', transformers.__version__)
print('sklearn     :', sklearn.__version__)
print('nltk        :', nltk.__version__)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 13.5 MB/s eta 0:00:00
torch       : 2.10.0+cu128 | cuda: True | Tesla T4
transformers: 4.57.6
sklearn     : 1.6.1
nltk        : 3.9.1


True

## 4. Run the full quantitative sweep

5 models x 3 splits (val / test / manual_200). On a T4 this should finish in ~3-6 minutes. Output CSV lands at `MISC/outputs/comparison/partD_quantitative.csv`.

In [4]:
!python evaluate_all.py

[info] loaded val: n=6391, toxic=9.2%
[info] loaded test_6w: n=6391, toxic=9.2%
[info] loaded manual_200: n=232, toxic=37.9%

[eval ] roberta (/content/drive/MyDrive/bravo11234/MISC/saved_model/roberta_base)
2026-04-21 15:16:43.589078: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776784603.611553    1847 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776784603.619501    1847 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776784603.640437    1847 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776784603.640476    1847 computation_plac

## 5. Show + download the merged table

In [5]:
import pandas as pd
csv_path = PROJECT / 'MISC' / 'outputs' / 'comparison' / 'partD_quantitative.csv'
df = pd.read_csv(csv_path)
print(df.to_string(index=False))

                  Model    Dataset    n  Accuracy  Precision  Recall  F1-score  F1-toxic  F1-macro
RoBERTa-base (weighted)        val 6391    0.9543     0.7182  0.8325    0.7712    0.7712    0.8729
RoBERTa-base (weighted)    test_6w 6391    0.9567     0.7336  0.8342    0.7807    0.7807    0.8783
RoBERTa-base (weighted) manual_200  232    0.6983     0.6607  0.4205    0.5139    0.5139    0.6476
             BERT-clean        val 6391    0.9601     0.7763  0.7986    0.7873    0.7873    0.8827
             BERT-clean    test_6w 6391    0.9598     0.7756  0.7953    0.7853    0.7853    0.8816
             BERT-clean manual_200  232    0.7069     0.7273  0.3636    0.4848    0.4848    0.6400
      BERT-aug+weighted        val 6391    0.9603     0.7722  0.8088    0.7901    0.7901    0.8841
      BERT-aug+weighted    test_6w 6391    0.9582     0.7547  0.8122    0.7824    0.7824    0.8796
      BERT-aug+weighted manual_200  232    0.7931     0.8125  0.5909    0.6842    0.6842    0.7652
          

In [6]:
from google.colab import files
files.download(str(csv_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>